# 신제품 부정 경험 고위험군 예측 모델 (전면 재설계)

**연구 목적**: 기존 제품의 리뷰 감성분석 결과를 학습 라벨로 활용하여, 리뷰가 없는 신제품의
**성분표 + 제품명 + 피부타입 정보만으로** 각 피부타입에서 부정적 사용경험 고위험군에 속할
가능성을 예측합니다.

**핵심 원칙**
- 리뷰 텍스트/리뷰 통계는 절대 X(입력)에 사용하지 않습니다 — Y(정답)를 만드는 데만 사용합니다.
- Y는 `groupby(["상품번호", "피부타입"])`로 **제품×피부타입** 단위로 만듭니다 (제품 단위 X 7반복 금지).
- V1(전체 전성분)과 V2(빈도필터 성분)는 **완전히 동일한 표본과 동일한 Fold**로 비교합니다.
- 모든 Train/Validation 분할(외부 CV, 내부 CV, Optuna, 임계값 선택, 조기종료)은 **product_id 그룹을 지킵니다.**
- 평가는 Accuracy/ROC-AUC 대신 **Precision, F0.5-score**를 최우선으로 봅니다.
> 이 파일은 팀원이 수행한 대안 모델링 실험입니다. 최종 채택 모델은 `notebooks/05_modeling/10_train_and_evaluate.ipynb`에서 확인합니다.


## 1. 패키지 설치 및 환경 설정


In [ ]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / 'data').is_dir() and (path / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.')

PROJECT_ROOT = find_project_root()
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXPERIMENT_RESULTS_DIR = (
    PROJECT_ROOT / 'reports' / 'experiments' / 'high_risk_prediction_redesign'
)
EXPERIMENT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
import os
import re
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, accuracy_score,
    confusion_matrix, precision_recall_curve
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 60)

print("환경 설정 완료")

## 2. 설정값


In [ ]:
# ===== 파일 경로 =====
V1_PATH = DATA_PROCESSED_DIR / '최종_V1_전체성분_V3V4V5.csv'
V2_PATH = DATA_PROCESSED_DIR / '최종_V2_선택성분_V3V4V5.csv'
REVIEW_PATH = DATA_PROCESSED_DIR / 'Gemini_리뷰감성분석_배송제외_최종.csv'
RESULTS_DIR = EXPERIMENT_RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)

# ===== 최소 리뷰 수 필터 =====
MIN_REVIEWS_PER_GROUP = 5   # 제품×피부타입 조합의 최소 리뷰 수 (1, 3, 5 등으로 조정 가능)

# ===== 고위험군 라벨 정의 =====
RISK_THRESHOLD_MODE = "quantile"     # "quantile" 또는 "fixed"
RISK_QUANTILE = 0.75                 # 부정비율 상위 25% = 75% 분위수
FIXED_RISK_THRESHOLD = 0.082         # RISK_THRESHOLD_MODE == "fixed"일 때 사용

# ===== 교차검증 설정 =====
N_SPLITS = 5
RANDOM_STATE = 42

# ===== 임계값 선택 =====
MIN_RECALL_FOR_THRESHOLD = 0.30
THRESHOLD_GRID = [round(x, 2) for x in np.arange(0.05, 0.96, 0.05)]

# ===== Optuna 튜닝 =====
RUN_OPTUNA = True
N_TRIALS = 30
OPTUNA_OBJECTIVE = "f05_pr_auc"   # "f05_pr_auc": 0.7*F0.5 + 0.3*PR-AUC / "f05_only": F0.5만 사용

print("설정값 로드 완료")
print(f"MIN_REVIEWS_PER_GROUP = {MIN_REVIEWS_PER_GROUP}")
print(f"RISK_THRESHOLD_MODE = {RISK_THRESHOLD_MODE} (RISK_QUANTILE={RISK_QUANTILE}, FIXED={FIXED_RISK_THRESHOLD})")
print(f"N_SPLITS = {N_SPLITS}, RANDOM_STATE = {RANDOM_STATE}")
print(f"RUN_OPTUNA = {RUN_OPTUNA}, N_TRIALS = {N_TRIALS}, OPTUNA_OBJECTIVE = {OPTUNA_OBJECTIVE}")

## 3. 데이터 로드


In [ ]:
df_v1_raw = pd.read_csv(V1_PATH, encoding='utf-8-sig')
df_v2_raw = pd.read_csv(V2_PATH, encoding='utf-8-sig')
df_review_raw = pd.read_csv(REVIEW_PATH, encoding='utf-8-sig')

print("V1 (전체 전성분) shape:", df_v1_raw.shape)
print("V2 (빈도필터 성분) shape:", df_v2_raw.shape)
print("리뷰 데이터 shape:", df_review_raw.shape)

print("\n[V1 컬럼 전체 목록]")
print(df_v1_raw.columns.tolist())
print("\n[V2 컬럼 전체 목록]")
print(df_v2_raw.columns.tolist())
print("\n[리뷰 데이터 컬럼 전체 목록]")
print(df_review_raw.columns.tolist())

## 4. 컬럼 검사


In [ ]:
# ---- 리뷰 데이터에 피부타입 컬럼이 있는지 검사 ----
SKIN_TYPE_CANDIDATES = ['피부타입', 'skin_type', '스킨타입', '피부 타입']
found_skin_col = None
for cand in SKIN_TYPE_CANDIDATES:
    if cand in df_review_raw.columns:
        found_skin_col = cand
        break

if found_skin_col is None:
    raise ValueError("리뷰 데이터에 피부타입 컬럼이 없어 제품×피부타입별 Y를 만들 수 없습니다.")

print(f"리뷰 데이터에서 피부타입 컬럼으로 '{found_skin_col}'을(를) 사용합니다.")

# ---- 필수 컬럼 존재 검사 ----
required_review_cols = ['상품번호', found_skin_col, '리뷰분류_1_0']
missing = [c for c in required_review_cols if c not in df_review_raw.columns]
if missing:
    raise ValueError(f"리뷰 데이터에 필수 컬럼이 없습니다: {missing}")

# ---- 리뷰분류_1_0이 0/1로만 구성되었는지 검사 ----
unique_vals = set(df_review_raw['리뷰분류_1_0'].dropna().unique().tolist())
if not unique_vals.issubset({0, 1}):
    raise ValueError(f"리뷰분류_1_0에 0/1이 아닌 값이 있습니다: {unique_vals}")
print("리뷰분류_1_0 값 검사 통과 (0/1로만 구성됨)")

# ---- V1, V2 접두사 검사 ----
def check_prefix_exists(df, prefix, name):
    cols = [c for c in df.columns if c.startswith(prefix)]
    if len(cols) == 0:
        raise ValueError(f"{name}에 '{prefix}' 접두사 컬럼이 없습니다.")
    return cols

v1_v1cols = check_prefix_exists(df_v1_raw, 'v1_', 'V1')
v1_v3cols = check_prefix_exists(df_v1_raw, 'v3_', 'V1')
v1_v4cols = check_prefix_exists(df_v1_raw, 'v4_', 'V1')
v1_v5cols = check_prefix_exists(df_v1_raw, 'v5_', 'V1')
v1_skincols = check_prefix_exists(df_v1_raw, 'skin_', 'V1')

v2_v2cols = check_prefix_exists(df_v2_raw, 'v2_', 'V2')
v2_v3cols = check_prefix_exists(df_v2_raw, 'v3_', 'V2')
v2_v4cols = check_prefix_exists(df_v2_raw, 'v4_', 'V2')
v2_v5cols = check_prefix_exists(df_v2_raw, 'v5_', 'V2')
v2_skincols = check_prefix_exists(df_v2_raw, 'skin_', 'V2')

print(f"V1: v1_*={len(v1_v1cols)}, v3_*={len(v1_v3cols)}, v4_*={len(v1_v4cols)}, v5_*={len(v1_v5cols)}, skin_*={len(v1_skincols)}")
print(f"V2: v2_*={len(v2_v2cols)}, v3_*={len(v2_v3cols)}, v4_*={len(v2_v4cols)}, v5_*={len(v2_v5cols)}, skin_*={len(v2_skincols)}")

for c in ['product_id', 'product_name', '피부타입', 'formula_group']:
    if c not in df_v1_raw.columns:
        raise ValueError(f"V1에 관리 컬럼 '{c}'가 없습니다.")
    if c not in df_v2_raw.columns:
        raise ValueError(f"V2에 관리 컬럼 '{c}'가 없습니다.")
print("V1/V2 필수 관리 컬럼 존재 확인 완료")

## 5. 제품×피부타입 Y 생성


In [ ]:
df_review = df_review_raw.copy()
df_review = df_review.rename(columns={'상품번호': 'product_id', found_skin_col: '피부타입'})
df_review['product_id'] = df_review['product_id'].astype(str).str.strip()
df_review['피부타입'] = df_review['피부타입'].astype(str).str.strip()

print("리뷰 원본 행 수:", len(df_review))

y_group = (
    df_review
    .groupby(['product_id', '피부타입'])
    .agg(
        y_전체리뷰=('리뷰분류_1_0', 'count'),
        y_긍정=('리뷰분류_1_0', 'sum'),
    )
    .reset_index()
)
y_group['y_부정'] = y_group['y_전체리뷰'] - y_group['y_긍정']
y_group['y_부정비율'] = y_group['y_부정'] / y_group['y_전체리뷰']

print("제품×피부타입 Y 행 수:", len(y_group))

dup_keys = y_group.duplicated(['product_id', '피부타입']).sum()
if dup_keys > 0:
    raise ValueError(f"제품×피부타입 Y 키 중복이 {dup_keys}건 있습니다.")
print("제품×피부타입 Y 키 중복 없음 확인")

print("\n리뷰 수 분포:")
print(y_group['y_전체리뷰'].describe())
print("\n피부타입별 관측치 수:")
print(y_group['피부타입'].value_counts())
print("\n피부타입별 평균 부정률:")
print(y_group.groupby('피부타입')['y_부정비율'].mean().sort_values(ascending=False))

## 6. 최소 리뷰 수 필터


In [ ]:
n_before = len(y_group)
y_filtered = y_group[y_group['y_전체리뷰'] >= MIN_REVIEWS_PER_GROUP].copy().reset_index(drop=True)
n_after = len(y_filtered)

print(f"필터링 전 제품×피부타입 수: {n_before}")
print(f"필터링 후 제품×피부타입 수: {n_after}")
print(f"제외된 행 수: {n_before - n_after} ({(n_before - n_after) / n_before * 100:.2f}%)")

print("\n피부타입별 남은 행 수:")
print(y_filtered['피부타입'].value_counts())

## 7. 고위험군 라벨 생성


In [ ]:
if RISK_THRESHOLD_MODE == "quantile":
    risk_threshold = y_filtered['y_부정비율'].quantile(RISK_QUANTILE)
elif RISK_THRESHOLD_MODE == "fixed":
    risk_threshold = FIXED_RISK_THRESHOLD
else:
    raise ValueError(f"알 수 없는 RISK_THRESHOLD_MODE: {RISK_THRESHOLD_MODE}")

print(f"RISK_THRESHOLD_MODE = {RISK_THRESHOLD_MODE}")
print(f"제품×피부타입 기준 75% 분위수: {y_filtered['y_부정비율'].quantile(0.75):.4f}")
print(f"고위험군 기준 부정률(risk_threshold): {risk_threshold:.4f}")

y_filtered['high_risk'] = (y_filtered['y_부정비율'] >= risk_threshold).astype(int)

n_risk = (y_filtered['high_risk'] == 1).sum()
n_safe = (y_filtered['high_risk'] == 0).sum()
print(f"고위험군 행 수: {n_risk}")
print(f"일반군 행 수: {n_safe}")
print(f"고위험군 비율: {n_risk / len(y_filtered) * 100:.2f}%")

print("\n피부타입별 고위험군 비율:")
print(y_filtered.groupby('피부타입')['high_risk'].mean().sort_values(ascending=False))

risk_label_summary = y_filtered.groupby('피부타입').agg(
    n=('high_risk', 'size'), risk_rate=('high_risk', 'mean')
).reset_index()
risk_label_summary.to_csv(os.path.join(RESULTS_DIR, 'risk_label_summary.csv'), index=False, encoding='utf-8-sig')

excluded = y_group[~y_group.set_index(['product_id', '피부타입']).index.isin(
    y_filtered.set_index(['product_id', '피부타입']).index
)]
excluded.to_csv(os.path.join(RESULTS_DIR, 'excluded_low_review_groups.csv'), index=False, encoding='utf-8-sig')
print(f"\n제외된 저리뷰 그룹 {len(excluded)}건을 excluded_low_review_groups.csv로 저장")

## 8. V1·V2 병합 및 공통 키 정렬


In [ ]:
def prepare_x_base(df_raw, name):
    df = df_raw.copy()
    df['product_id'] = df['product_id'].astype(str).str.strip()
    df['피부타입'] = df['피부타입'].astype(str).str.strip()
    dup = df.duplicated(['product_id', '피부타입']).sum()
    if dup > 0:
        raise ValueError(f"{name} product_id+피부타입 키 중복이 {dup}건 있습니다.")
    return df

df_v1 = prepare_x_base(df_v1_raw, 'V1')
df_v2 = prepare_x_base(df_v2_raw, 'V2')
print("V1 product_id+피부타입 키 중복 없음 확인")
print("V2 product_id+피부타입 키 중복 없음 확인")

n_v1_before, n_v2_before = len(df_v1), len(df_v2)

merged_v1 = pd.merge(df_v1, y_filtered, on=['product_id', '피부타입'], how='inner', validate='one_to_one')
merged_v2 = pd.merge(df_v2, y_filtered, on=['product_id', '피부타입'], how='inner', validate='one_to_one')

print(f"V1 병합 전 행 수: {n_v1_before}, 병합 후 행 수: {len(merged_v1)}")
print(f"V2 병합 전 행 수: {n_v2_before}, 병합 후 행 수: {len(merged_v2)}")

x_v1_excluded = n_v1_before - len(merged_v1)
x_v2_excluded = n_v2_before - len(merged_v2)
y_excluded_from_v1 = len(y_filtered) - len(merged_v1)
y_excluded_from_v2 = len(y_filtered) - len(merged_v2)
print(f"Y와 매칭되지 않아 제외된 V1 X 행 수: {x_v1_excluded}")
print(f"Y와 매칭되지 않아 제외된 V2 X 행 수: {x_v2_excluded}")
print(f"V1과 매칭되지 않아 제외된 Y 행 수: {y_excluded_from_v1}")
print(f"V2와 매칭되지 않아 제외된 Y 행 수: {y_excluded_from_v2}")

# ---- V1과 V2의 공통 키만 사용 ----
keys_v1 = set(zip(merged_v1['product_id'], merged_v1['피부타입']))
keys_v2 = set(zip(merged_v2['product_id'], merged_v2['피부타입']))
common_keys = keys_v1 & keys_v2
print(f"\nV1 키 수: {len(keys_v1)}, V2 키 수: {len(keys_v2)}, 공통 키 수: {len(common_keys)}")

common_keys_df = pd.DataFrame(list(common_keys), columns=['product_id', '피부타입'])
common_keys_df = common_keys_df.sort_values(['product_id', '피부타입']).reset_index(drop=True)
common_keys_df.to_csv(os.path.join(RESULTS_DIR, 'v1_v2_common_keys.csv'), index=False, encoding='utf-8-sig')

merged_v1 = pd.merge(merged_v1, common_keys_df, on=['product_id', '피부타입'], how='inner')
merged_v2 = pd.merge(merged_v2, common_keys_df, on=['product_id', '피부타입'], how='inner')

merged_v1 = merged_v1.sort_values(['product_id', '피부타입']).reset_index(drop=True)
merged_v2 = merged_v2.sort_values(['product_id', '피부타입']).reset_index(drop=True)

# ---- 일치 검증 ----
assert len(merged_v1) == len(merged_v2), "V1과 V2 행 수가 다릅니다."
assert (merged_v1['product_id'].values == merged_v2['product_id'].values).all(), "V1과 V2의 product_id 순서가 다릅니다."
assert (merged_v1['피부타입'].values == merged_v2['피부타입'].values).all(), "V1과 V2의 피부타입 순서가 다릅니다."
assert (merged_v1['high_risk'].values == merged_v2['high_risk'].values).all(), "V1과 V2의 high_risk 순서가 다릅니다."
assert np.allclose(merged_v1['y_부정비율'].values, merged_v2['y_부정비율'].values), "V1과 V2의 y_부정비율이 다릅니다."
assert merged_v1.duplicated(['product_id', '피부타입']).sum() == 0, "V1에 중복 키가 있습니다."
assert merged_v2.duplicated(['product_id', '피부타입']).sum() == 0, "V2에 중복 키가 있습니다."

print(f"\n최종 공통 표본 행 수: {len(merged_v1)}")
print("V1/V2 일치 검증 통과")

## 9. X 피처셋 정의 및 안전한 컬럼명 변환

동일 표본·동일 Fold로 **4개 피처셋**을 비교합니다.

| 피처셋 | 구성 | 목적 |
|---|---|---|
| A_피부타입만 | `skin_*` | 성분이 실제로 신호를 더하는지 판단하는 베이스라인 |
| B_핵심60_V345 | `skin_* + v3_* + v4_* + v5_*` | 기능군·근거·테마 요약 변수만 (저차원) |
| C_V2빈도필터성분 | `skin_* + v2_* + v3_* + v4_* + v5_*` | 빈도필터 성분 원핫 포함 |
| D_V1전체전성분 | `skin_* + v1_* + v3_* + v4_* + v5_*` | 전체 전성분 원핫 포함 |

A를 넘어서지 못하면 성분 정보가 기여하지 않는다는 뜻이고,
B가 C·D와 비슷하면 고차원 성분 원핫이 불필요하다는 근거가 됩니다.


In [ ]:
MANAGEMENT_COLS = ['product_id', 'product_name', '피부타입', 'formula_group']
Y_LEAKAGE_COLS = ['y_전체리뷰', 'y_긍정', 'y_부정', 'y_부정비율', 'high_risk']

def select_feature_columns(df, allowed_prefixes):
    cols = []
    for c in df.columns:
        if c in MANAGEMENT_COLS or c in Y_LEAKAGE_COLS:
            continue
        if any(c.startswith(p) for p in allowed_prefixes):
            cols.append(c)
    return cols


def make_safe_column_names(columns, mapping_save_path):
    """특수문자를 '_'로 치환하고, 충돌 시 suffix를 붙여 원본-변환 매핑을 저장"""
    safe_names = []
    used = {}
    mapping_rows = []
    for orig in columns:
        safe = re.sub(r'[^a-zA-Z0-9\uac00-\ud7a3]', '_', str(orig))
        base_safe = safe
        cnt = used.get(base_safe, 0)
        if cnt > 0:
            safe = f"{base_safe}__dup{cnt}"
        used[base_safe] = cnt + 1
        safe_names.append(safe)
        mapping_rows.append({'original_column': orig, 'safe_column': safe})

    mapping_df = pd.DataFrame(mapping_rows)
    dup_safe = mapping_df['safe_column'].duplicated().sum()
    if dup_safe > 0:
        raise ValueError(f"안전한 컬럼명 변환 후에도 {dup_safe}건의 충돌이 남아있습니다.")

    mapping_df.to_csv(mapping_save_path, index=False, encoding='utf-8-sig')
    return safe_names, mapping_df


# =====================================================================
# 4개 피처셋을 동일 표본/동일 Fold로 비교
#  A: 피부타입만          -> 성분이 신호를 더하는지 판단하기 위한 베이스라인
#  B: 핵심 60 (skin+v3v4v5) -> 기능군/근거/테마 요약 변수만
#  C: V2 주성분 블록
#  D: V1 전성분 블록
# =====================================================================
DATASET_SPECS = {
    'A_피부타입만':      {'source': 'v1', 'prefixes': ('skin_',)},
    'B_핵심60_V345':     {'source': 'v1', 'prefixes': ('skin_', 'v3_', 'v4_', 'v5_')},
    'C_V2빈도필터성분':   {'source': 'v2', 'prefixes': ('skin_', 'v2_', 'v3_', 'v4_', 'v5_')},
    'D_V1전체전성분':     {'source': 'v1', 'prefixes': ('skin_', 'v1_', 'v3_', 'v4_', 'v5_')},
}

DATASETS = {}


In [ ]:
for ds_name, spec in DATASET_SPECS.items():
    src_df = merged_v1 if spec['source'] == 'v1' else merged_v2
    feat_cols = select_feature_columns(src_df, spec['prefixes'])

    # 교차 오염 검사
    if spec['source'] == 'v1':
        assert not any(c.startswith('v2_') for c in feat_cols), f"{ds_name}에 v2_* 컬럼이 섞여 있습니다."
    else:
        assert not any(c.startswith('v1_') for c in feat_cols), f"{ds_name}에 v1_* 컬럼이 섞여 있습니다."
    for c in MANAGEMENT_COLS + Y_LEAKAGE_COLS:
        assert c not in feat_cols, f"{ds_name} 피처에 {c}가 포함되어 있습니다."

    safe_names, mapping_df = make_safe_column_names(
        feat_cols, os.path.join(RESULTS_DIR, f'feature_name_mapping_{ds_name}.csv')
    )
    X_ds = src_df[feat_cols].copy()
    X_ds.columns = safe_names

    non_numeric = X_ds.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        raise ValueError(f"{ds_name} X에 숫자형이 아닌 컬럼이 있습니다: {non_numeric}")
    if X_ds.isna().sum().sum() > 0:
        raise ValueError(f"{ds_name} X에 결측값이 있습니다.")

    DATASETS[ds_name] = {
        'X': X_ds, 'feature_cols_original': feat_cols,
        'safe_names': safe_names, 'mapping': mapping_df,
    }
    print(f"{ds_name:<20} 피처 수 = {X_ds.shape[1]:>5}")

# V1/V2 공통 블록 값 일치 검사
common_block_cols = [c for c in merged_v1.columns if c.startswith(('skin_', 'v3_', 'v4_', 'v5_'))]
mismatch = sum(
    1 for c in common_block_cols
    if c in merged_v2.columns
    and not np.allclose(merged_v1[c].values.astype(float), merged_v2[c].values.astype(float))
)
if mismatch > 0:
    raise ValueError(f"V1/V2 공통 블록 중 {mismatch}개 컬럼 값이 일치하지 않습니다.")
print(f"\nV1/V2 공통 블록({len(common_block_cols)}개 컬럼) 값 일치 확인")

y_all = merged_v1['high_risk'].values
y_ratio_all = merged_v1['y_부정비율'].values
product_id_groups = merged_v1['product_id'].values

assert pd.isna(y_all).sum() == 0, "Y에 결측값이 있습니다."
print("X 숫자형/결측 0, Y 결측 0 확인")

X_reference = DATASETS['D_V1전체전성분']['X']   # Fold 생성 기준 (어느 것을 써도 행 순서 동일)
print(f"\n최종 표본: {len(y_all)}행 / 제품 {len(set(product_id_groups))}개 / 고위험군 비율 {y_all.mean():.4f}")


## 10. 공통 Fold 생성 (V1·V2·모든 모델이 공유)


In [ ]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_reference, y_all, groups=product_id_groups))

print(f"{N_SPLITS}-Fold 생성 완료 (4개 피처셋 x 모든 모델이 이 folds를 공유)")

fold_product_lists = []
for i, (tr_idx, va_idx) in enumerate(folds):
    tr_products = set(product_id_groups[tr_idx])
    va_products = set(product_id_groups[va_idx])
    overlap = tr_products & va_products
    assert len(overlap) == 0, f"Fold {i}: train/validation product_id 교집합이 {len(overlap)}건 있습니다."

    print(f"Fold {i}: train 행={len(tr_idx)}(제품 {len(tr_products)}개, 위험비율 {y_all[tr_idx].mean():.3f}) "
          f"/ valid 행={len(va_idx)}(제품 {len(va_products)}개, 위험비율 {y_all[va_idx].mean():.3f})")

    fold_product_lists.append({
        'fold': i,
        'train_products': sorted(tr_products),
        'valid_products': sorted(va_products),
    })

with open(os.path.join(RESULTS_DIR, 'fold_product_lists.json'), 'w', encoding='utf-8') as f:
    json.dump(fold_product_lists, f, ensure_ascii=False, indent=2)

print("\n모든 Fold에서 train/validation product_id 교집합 0 확인 (누수 없음)")

## 11. 평가 함수


In [ ]:
def compute_topk_metrics(y_true, y_prob, top_ratio=0.25):
    """예측확률 상위 top_ratio 비율을 고위험 예측군으로 지정하고 Precision/Recall/Lift 계산"""
    n = len(y_true)
    k = max(1, int(np.ceil(n * top_ratio)))
    order = np.argsort(-y_prob)
    top_idx = order[:k]

    top_true = y_true[top_idx]
    precision_at_k = top_true.mean() if k > 0 else np.nan
    overall_risk_rate = y_true.mean()
    recall_at_k = top_true.sum() / y_true.sum() if y_true.sum() > 0 else np.nan
    lift_at_k = precision_at_k / overall_risk_rate if overall_risk_rate > 0 else np.nan

    return precision_at_k, recall_at_k, lift_at_k


def evaluate_predictions(y_true, y_prob, y_pred):
    """Precision, F0.5, Recall, PR-AUC, F1, ROC-AUC, Accuracy, Top25% 지표를 한번에 계산"""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = np.asarray(y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f05 = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else np.nan
    acc = accuracy_score(y_true, y_pred)

    p25, r25, lift25 = compute_topk_metrics(y_true, y_prob, top_ratio=0.25)

    return {
        'Precision': precision, 'F0.5': f05, 'Recall': recall,
        'PR_AUC': pr_auc, 'F1': f1, 'ROC_AUC': roc_auc, 'Accuracy': acc,
        'Precision_at_25': p25, 'Recall_at_25': r25, 'Lift_at_25': lift25,
    }

print("평가 함수 정의 완료")

## 12. 임계값 선택 함수


In [ ]:
def select_threshold(y_true, y_prob, thresholds=None, min_recall=MIN_RECALL_FOR_THRESHOLD):
    """외부 Train 내부(Threshold-Validation)에서 F0.5 기준으로 최적 임계값 선택.
    Recall >= min_recall 조건을 만족하는 임계값 중 F0.5가 가장 높은 것을 선택.
    조건을 만족하는 임계값이 없으면 F0.5가 가장 높은 임계값을 선택."""
    if thresholds is None:
        thresholds = THRESHOLD_GRID

    records = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f05 = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)
        records.append({'threshold': t, 'precision': precision, 'recall': recall, 'f05': f05})

    rec_df = pd.DataFrame(records)
    candidates = rec_df[rec_df['recall'] >= min_recall]

    if len(candidates) == 0:
        candidates = rec_df

    max_f05 = candidates['f05'].max()
    candidates = candidates[candidates['f05'] == max_f05]
    max_precision = candidates['precision'].max()
    candidates = candidates[candidates['precision'] == max_precision]
    best_threshold = candidates['threshold'].max()

    return float(best_threshold)


def fit_predict_with_group_threshold_split(model, X_train, y_train, groups_train,
                                            X_valid, threshold_valid_size=0.2, random_state=RANDOM_STATE):
    """외부 Train 데이터를 다시 product_id 그룹 기준으로 Train/Threshold-Validation으로 나눠서
    임계값을 외부 Validation을 보지 않고 선택"""
    gss = GroupShuffleSplit(n_splits=1, test_size=threshold_valid_size, random_state=random_state)
    fit_idx, thr_idx = next(gss.split(X_train, y_train, groups=groups_train))

    fit_products = set(np.asarray(groups_train)[fit_idx])
    thr_products = set(np.asarray(groups_train)[thr_idx])
    assert len(fit_products & thr_products) == 0, "임계값 선택용 내부 분할에서 product_id가 겹칩니다."

    X_fit, X_thr = X_train.iloc[fit_idx], X_train.iloc[thr_idx]
    y_fit, y_thr = np.asarray(y_train)[fit_idx], np.asarray(y_train)[thr_idx]

    model.fit(X_fit, y_fit)
    thr_prob = model.predict_proba(X_thr)[:, 1]
    best_threshold = select_threshold(y_thr, thr_prob)

    valid_prob = model.predict_proba(X_valid)[:, 1]

    return best_threshold, valid_prob, model

print("임계값 선택 함수 정의 완료")

## 13. 기본 모델 정의


In [ ]:
def get_model_spec(model_name, scale_pos_weight, tuned_params=None, random_state=RANDOM_STATE):
    """model_name별로 (ModelClass, params dict)를 반환.
    tuned_params가 주어지면 Optuna로 찾은 값으로 덮어쓰고, 없으면 baseline 기본값을 사용."""

    if model_name == 'LogisticRegression':
        params = dict(penalty='l2', C=1.0, max_iter=5000,
                      class_weight='balanced', random_state=random_state, solver='liblinear')
        if tuned_params:
            params['C'] = tuned_params.get('C', params['C'])
            params['penalty'] = tuned_params.get('penalty', params['penalty'])
        return LogisticRegression, params

    if model_name == 'RandomForest':
        params = dict(n_estimators=400, max_depth=8, min_samples_leaf=2, max_features='sqrt',
                      class_weight={0: 1.0, 1: float(scale_pos_weight)},
                      random_state=random_state, n_jobs=-1)
        if tuned_params:
            for k in ['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features']:
                if k in tuned_params:
                    params[k] = tuned_params[k]
        return RandomForestClassifier, params

    if model_name == 'XGBoost':
        params = dict(n_estimators=300, max_depth=3, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                      scale_pos_weight=scale_pos_weight,
                      eval_metric='logloss', random_state=random_state)
        if tuned_params:
            for k in ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'reg_lambda']:
                if k in tuned_params:
                    params[k] = tuned_params[k]
        return XGBClassifier, params

    if model_name == 'CatBoost':
        params = dict(iterations=300, depth=3, learning_rate=0.05, l2_leaf_reg=3.0,
                      scale_pos_weight=scale_pos_weight,
                      eval_metric='Logloss', random_seed=random_state, verbose=0)
        if tuned_params:
            for k in ['iterations', 'depth', 'learning_rate', 'l2_leaf_reg']:
                if k in tuned_params:
                    params[k] = tuned_params[k]
        return CatBoostClassifier, params

    raise ValueError(f"알 수 없는 model_name: {model_name}")

BASE_MODEL_NAMES = ['LogisticRegression', 'RandomForest', 'XGBoost', 'CatBoost']
EARLY_STOPPING_MODELS = ('XGBoost', 'CatBoost')

print("기본 모델 정의(get_model_spec) 완료. 모델:", BASE_MODEL_NAMES)

## 14. 5-Fold 그룹 교차검증 (공통 함수: baseline/tuned 재사용)


In [ ]:
def run_group_cv(X, y, groups, dataset_name, folds, model_names,
                  tuned_params_dict=None, stage_label='baseline'):
    """공통 folds를 사용해 모델들을 5-Fold 그룹 교차검증으로 평가.
    각 Fold의 Train 내부에서만 scale_pos_weight와 임계값을 계산 (Validation 정보 미사용).
    XGBoost/CatBoost는 조기종료용 내부 검증셋도 product_id 그룹을 지켜서 분리."""
    tuned_params_dict = tuned_params_dict or {}
    fold_records = []
    oof_records = []

    for fold_i, (tr_idx, va_idx) in enumerate(folds):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        groups_tr = groups[tr_idx]

        n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

        for model_name in model_names:
            tuned_params = tuned_params_dict.get(model_name)
            ModelClass, params = get_model_spec(model_name, scale_pos_weight, tuned_params=tuned_params)

            if model_name in EARLY_STOPPING_MODELS:
                gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
                sub_tr_idx, sub_va_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))

                sub_tr_products = set(np.asarray(groups_tr)[sub_tr_idx])
                sub_va_products = set(np.asarray(groups_tr)[sub_va_idx])
                assert len(sub_tr_products & sub_va_products) == 0, \
                    f"Fold {fold_i} {model_name} 조기종료 내부 분할에서 product_id가 겹칩니다."

                X_sub_tr, X_sub_va = X_tr.iloc[sub_tr_idx], X_tr.iloc[sub_va_idx]
                y_sub_tr, y_sub_va = y_tr[sub_tr_idx], y_tr[sub_va_idx]

                model = ModelClass(**params, early_stopping_rounds=30)
                if model_name == 'XGBoost':
                    model.fit(X_sub_tr, y_sub_tr, eval_set=[(X_sub_va, y_sub_va)], verbose=False)
                else:
                    model.fit(X_sub_tr, y_sub_tr, eval_set=(X_sub_va, y_sub_va), use_best_model=True)

                thr_prob = model.predict_proba(X_sub_va)[:, 1]
                best_threshold = select_threshold(y_sub_va, thr_prob)
            else:
                model = ModelClass(**params)
                best_threshold, _, model = fit_predict_with_group_threshold_split(
                    model, X_tr, y_tr, groups_tr, X_va
                )

            va_prob = model.predict_proba(X_va)[:, 1]
            va_pred = (va_prob >= best_threshold).astype(int)

            metrics = evaluate_predictions(y_va, va_prob, va_pred)
            metrics.update({
                'dataset': dataset_name, 'model': model_name, 'stage': stage_label, 'fold': fold_i,
                'threshold': best_threshold, 'n_train': len(tr_idx), 'n_valid': len(va_idx),
                'n_features': X.shape[1],
            })
            fold_records.append(metrics)

            for local_i, global_i in enumerate(va_idx):
                oof_records.append({
                    'product_id': product_id_groups[global_i],
                    'product_name': merged_v1['product_name'].values[global_i],
                    '피부타입': merged_v1['피부타입'].values[global_i],
                    'y_전체리뷰': merged_v1['y_전체리뷰'].values[global_i],
                    'y_부정': merged_v1['y_부정'].values[global_i],
                    'y_부정비율': merged_v1['y_부정비율'].values[global_i],
                    'high_risk': int(y_va[local_i]),
                    'dataset': dataset_name, 'model': model_name, 'stage': stage_label, 'fold': fold_i,
                    'predicted_probability': va_prob[local_i],
                    'selected_threshold': best_threshold,
                    'predicted_label': int(va_pred[local_i]),
                    'top25_selected': False,
                })

            print(f"[{dataset_name}][{stage_label}] Fold {fold_i} {model_name:18s} | "
                  f"Precision={metrics['Precision']:.3f} F0.5={metrics['F0.5']:.3f} "
                  f"Recall={metrics['Recall']:.3f} PR-AUC={metrics['PR_AUC']:.3f} "
                  f"ROC-AUC={metrics['ROC_AUC']:.3f} thr={best_threshold:.2f}")

    fold_df = pd.DataFrame(fold_records)
    oof_df = pd.DataFrame(oof_records)

    for (ds, model_name, stg, fold_i), grp in oof_df.groupby(['dataset', 'model', 'stage', 'fold']):
        k = max(1, int(np.ceil(len(grp) * 0.25)))
        top_idx = grp['predicted_probability'].sort_values(ascending=False).index[:k]
        oof_df.loc[top_idx, 'top25_selected'] = True

    return fold_df, oof_df


In [ ]:
print("run_group_cv 함수 정의 완료")


In [ ]:
print("=" * 80)
print("1단계: 고정 파라미터 5-Fold 그룹 교차검증 (4개 피처셋 x 4개 모델)")
print("=" * 80)

fold_dfs, oof_dfs = [], []
for ds_name, ds in DATASETS.items():
    fdf, odf = run_group_cv(
        ds['X'], y_all, product_id_groups, ds_name, folds, BASE_MODEL_NAMES, stage_label='baseline'
    )
    fold_dfs.append(fdf)
    oof_dfs.append(odf)

fold_df_baseline = pd.concat(fold_dfs, ignore_index=True)
oof_df_baseline = pd.concat(oof_dfs, ignore_index=True)

print(f"\nbaseline Fold별 결과 {len(fold_df_baseline)}행 생성 완료")

## 15. 결과 요약 (1단계: baseline)


In [ ]:
def summarize_fold_metrics(fold_df):
    summary = fold_df.groupby(['dataset', 'model', 'stage']).agg(
        n_features=('n_features', 'first'),
        Precision_mean=('Precision', 'mean'), Precision_std=('Precision', 'std'),
        F05_mean=('F0.5', 'mean'), F05_std=('F0.5', 'std'),
        Recall_mean=('Recall', 'mean'), Recall_std=('Recall', 'std'),
        F1_mean=('F1', 'mean'), F1_std=('F1', 'std'),
        PR_AUC_mean=('PR_AUC', 'mean'), PR_AUC_std=('PR_AUC', 'std'),
        ROC_AUC_mean=('ROC_AUC', 'mean'), ROC_AUC_std=('ROC_AUC', 'std'),
        Accuracy_mean=('Accuracy', 'mean'), Accuracy_std=('Accuracy', 'std'),
        Precision_at_25_mean=('Precision_at_25', 'mean'),
        Recall_at_25_mean=('Recall_at_25', 'mean'),
        Lift_at_25_mean=('Lift_at_25', 'mean'),
        Threshold_mean=('threshold', 'mean'), Threshold_std=('threshold', 'std'),
    ).reset_index()

    summary = summary.rename(columns={
        'dataset': '데이터셋', 'model': '모델', 'stage': '단계', 'n_features': 'Feature 수'
    })
    return summary


baseline_summary = summarize_fold_metrics(fold_df_baseline)
baseline_summary_f05 = baseline_summary.sort_values(
    ['F05_mean', 'Precision_mean', 'PR_AUC_mean'], ascending=False
).reset_index(drop=True)

print("[baseline: F0.5 기준 정렬]")
display(baseline_summary_f05)

## 16. Optuna 튜닝 (2단계)

1단계 baseline 비교 결과와 별개로, `RUN_OPTUNA=True`이면 4개 모델 × 2개 데이터셋 전부에 대해
Optuna 탐색을 실행합니다 (계산량이 많습니다 — 필요하면 `RUN_OPTUNA=False`로 끄거나 `N_TRIALS`를 줄이세요).

- 목적함수는 F1·ROC-AUC 조합이 아니라 **F0.5 (또는 0.7×F0.5 + 0.3×PR-AUC)** 를 사용
- 모든 trial의 교차검증도 `StratifiedGroupKFold` + `groups=product_id`로 동일한 `folds`를 사용
- V1과 V2는 동일한 분할 Seed·Fold 정책을 공유


In [ ]:
if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print("Optuna 튜닝을 실행합니다. 모델 수 x 데이터셋 수만큼 반복 학습하므로 계산량이 많을 수 있습니다.")
else:
    print("RUN_OPTUNA=False로 설정되어 있어 튜닝 관련 셀은 건너뜁니다.")

In [ ]:
def optuna_objective_score(y_true, y_prob, y_pred):
    f05 = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)
    pr_auc = average_precision_score(y_true, y_prob)
    if OPTUNA_OBJECTIVE == "f05_only":
        return f05
    elif OPTUNA_OBJECTIVE == "f05_pr_auc":
        return 0.7 * f05 + 0.3 * pr_auc
    else:
        raise ValueError(f"알 수 없는 OPTUNA_OBJECTIVE: {OPTUNA_OBJECTIVE}")


def cv_objective_for_params(build_model_fn, X, y, groups, folds):
    """주어진 모델 생성 함수로 공통 folds 전체에 대해 학습·평가하고 목적함수 평균 반환"""
    scores = []
    for tr_idx, va_idx in folds:
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

        model = build_model_fn(scale_pos_weight)
        model.fit(X_tr, y_tr)
        va_prob = model.predict_proba(X_va)[:, 1]
        va_pred = (va_prob >= 0.5).astype(int)

        scores.append(optuna_objective_score(y_va, va_prob, va_pred))

    return float(np.mean(scores))


In [ ]:
def tune_model_with_optuna(model_type, X, y, groups, folds, n_trials=None, random_state=RANDOM_STATE, name=''):
    """model_type: 'LogisticRegression' | 'RandomForest' | 'XGBoost' | 'CatBoost'"""
    n_trials = n_trials or N_TRIALS

    if model_type == 'LogisticRegression':
        def build_model(trial, scale_pos_weight):
            C = trial.suggest_float('C', 0.001, 10.0, log=True)
            penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
            return LogisticRegression(C=C, penalty=penalty, solver='liblinear', max_iter=5000,
                                       class_weight='balanced', random_state=random_state)

    elif model_type == 'RandomForest':
        def build_model(trial, scale_pos_weight):
            return RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 100, 400),
                max_depth=trial.suggest_int('max_depth', 3, 8),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 2, 10),
                max_features=trial.suggest_float('max_features', 0.3, 1.0),
                class_weight={0: 1.0, 1: float(scale_pos_weight)},
                random_state=random_state, n_jobs=-1
            )

    elif model_type == 'XGBoost':
        def build_model(trial, scale_pos_weight):
            return XGBClassifier(
                n_estimators=trial.suggest_int('n_estimators', 100, 400),
                max_depth=trial.suggest_int('max_depth', 2, 4),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                subsample=trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
                reg_lambda=trial.suggest_float('reg_lambda', 1.0, 10.0),
                scale_pos_weight=scale_pos_weight,
                eval_metric='logloss', random_state=random_state
            )

    elif model_type == 'CatBoost':
        def build_model(trial, scale_pos_weight):
            return CatBoostClassifier(
                iterations=trial.suggest_int('iterations', 100, 400),
                depth=trial.suggest_int('depth', 2, 4),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 3.0, 10.0),
                scale_pos_weight=scale_pos_weight,
                eval_metric='Logloss', random_seed=random_state, verbose=0
            )
    else:
        raise ValueError(f"알 수 없는 model_type: {model_type}")

    def objective(trial):
        def build_fn(scale_pos_weight):
            return build_model(trial, scale_pos_weight)
        return cv_objective_for_params(build_fn, X, y, groups, folds)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"[{name}] {model_type} Optuna 최적 목적함수({OPTUNA_OBJECTIVE}) 값: {study.best_value:.4f}")
    print(f"[{name}] {model_type} Best params: {study.best_params}")

    return study


In [ ]:
print("Optuna 튜닝 함수 정의 완료")


In [ ]:
tuned_params_all = {ds_name: {} for ds_name in DATASETS}

if RUN_OPTUNA:
    for ds_name, ds in DATASETS.items():
        for model_type in BASE_MODEL_NAMES:
            print(f"\n{'=' * 70}\n[{ds_name}] {model_type} Optuna 튜닝 시작 (n_trials={N_TRIALS})\n{'=' * 70}")
            study = tune_model_with_optuna(model_type, ds['X'], y_all, product_id_groups, folds, name=ds_name)
            tuned_params_all[ds_name][model_type] = study.best_params

    with open(os.path.join(RESULTS_DIR, 'tuned_params_all.json'), 'w', encoding='utf-8') as f:
        json.dump(tuned_params_all, f, ensure_ascii=False, indent=2)
    print("\nOptuna 튜닝 완료, tuned_params_all.json 저장")
else:
    print("RUN_OPTUNA=False로 설정되어 튜닝을 건너뜁니다.")

## 17. 최종 비교 (baseline vs tuned, V1 vs V2)


In [ ]:
if RUN_OPTUNA:
    tuned_fold_dfs, tuned_oof_dfs = [], []
    for ds_name, ds in DATASETS.items():
        fdf, odf = run_group_cv(
            ds['X'], y_all, product_id_groups, ds_name, folds, BASE_MODEL_NAMES,
            tuned_params_dict=tuned_params_all[ds_name], stage_label='tuned'
        )
        tuned_fold_dfs.append(fdf)
        tuned_oof_dfs.append(odf)

    fold_df_combined = pd.concat([fold_df_baseline] + tuned_fold_dfs, ignore_index=True)
    oof_df_combined = pd.concat([oof_df_baseline] + tuned_oof_dfs, ignore_index=True)
else:
    fold_df_combined = fold_df_baseline.copy()
    oof_df_combined = oof_df_baseline.copy()

fold_df_combined.to_csv(os.path.join(RESULTS_DIR, 'fold_metrics.csv'), index=False, encoding='utf-8-sig')
oof_df_combined.to_csv(os.path.join(RESULTS_DIR, 'oof_predictions.csv'), index=False, encoding='utf-8-sig')

threshold_results = fold_df_combined[[
    'dataset', 'model', 'stage', 'fold', 'threshold', 'Precision', 'Recall', 'F0.5', 'F1',
    'PR_AUC', 'ROC_AUC', 'Accuracy', 'Precision_at_25', 'Recall_at_25', 'Lift_at_25'
]]
threshold_results.to_csv(os.path.join(RESULTS_DIR, 'threshold_results.csv'), index=False, encoding='utf-8-sig')

print(f"fold_metrics.csv ({len(fold_df_combined)}행), oof_predictions.csv ({len(oof_df_combined)}행), "
      f"threshold_results.csv 저장 완료")

In [ ]:
summary_combined = summarize_fold_metrics(fold_df_combined)

summary_combined_f05 = summary_combined.sort_values(
    ['F05_mean', 'Precision_mean', 'PR_AUC_mean'], ascending=False
).reset_index(drop=True)
summary_combined_precision = summary_combined.sort_values(
    ['Precision_mean', 'F05_mean', 'PR_AUC_mean'], ascending=False
).reset_index(drop=True)

print("[F0.5 기준 최종 비교 (baseline + tuned)]")
display(summary_combined_f05)
print("\n[Precision 기준 최종 비교]")
display(summary_combined_precision)

summary_combined_f05.to_csv(os.path.join(RESULTS_DIR, 'model_comparison_summary.csv'), index=False, encoding='utf-8-sig')

best_row = summary_combined_f05.iloc[0]
best_model_info = {
    'dataset': best_row['데이터셋'], 'model': best_row['모델'], 'stage': best_row['단계'],
    'F05_mean': float(best_row['F05_mean']), 'Precision_mean': float(best_row['Precision_mean']),
    'PR_AUC_mean': float(best_row['PR_AUC_mean']), 'ROC_AUC_mean': float(best_row['ROC_AUC_mean']),
    'Threshold_mean': float(best_row['Threshold_mean']),
}
with open(os.path.join(RESULTS_DIR, 'best_model_info.json'), 'w', encoding='utf-8') as f:
    json.dump(best_model_info, f, ensure_ascii=False, indent=2)

print("\n최종 최우수 모델:", best_model_info)

## 18. OOF 예측 기준 종합 성능


In [ ]:
oof_summary_records = []
for (ds, mdl, stg), grp in oof_df_combined.groupby(['dataset', 'model', 'stage']):
    y_true = grp['high_risk'].values
    y_prob = grp['predicted_probability'].values
    y_pred = grp['predicted_label'].values
    m = evaluate_predictions(y_true, y_prob, y_pred)
    m.update({'dataset': ds, 'model': mdl, 'stage': stg})
    oof_summary_records.append(m)

oof_summary_df = pd.DataFrame(oof_summary_records).sort_values('F0.5', ascending=False).reset_index(drop=True)
print("[OOF 기준 데이터셋x모델x단계별 종합 성능]")
display(oof_summary_df)
oof_summary_df.to_csv(os.path.join(RESULTS_DIR, 'oof_summary.csv'), index=False, encoding='utf-8-sig')

## 19. 필수 시각화


In [ ]:
plot_df = summary_combined_f05.copy()
plot_labels = plot_df['데이터셋'] + '\n' + plot_df['모델'] + '(' + plot_df['단계'] + ')'

# 1. 모델별 Precision / F0.5 / Recall / PR-AUC 비교 막대그래프
fig, ax = plt.subplots(figsize=(15, 6))
x = np.arange(len(plot_df))
width = 0.2
ax.bar(x - 1.5 * width, plot_df['Precision_mean'], width, label='Precision')
ax.bar(x - 0.5 * width, plot_df['F05_mean'], width, label='F0.5')
ax.bar(x + 0.5 * width, plot_df['Recall_mean'], width, label='Recall')
ax.bar(x + 1.5 * width, plot_df['PR_AUC_mean'], width, label='PR-AUC')
ax.set_xticks(x)
ax.set_xticklabels(plot_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_title('데이터셋 x 모델 x 단계별 Precision / F0.5 / Recall / PR-AUC 비교')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig1_model_metric_comparison.png'), dpi=150)
plt.show()

In [ ]:
# 2. 피처셋별 비교 (성분 정보가 피부타입 베이스라인을 넘어서는가)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pivot_f05 = plot_df.pivot_table(index='데이터셋', columns='모델', values='F05_mean', aggfunc='mean')
pivot_f05.plot(kind='bar', ax=axes[0])
axes[0].set_title('피처셋 x 모델별 F0.5 평균')
axes[0].set_ylabel('F0.5 (mean)')
axes[0].tick_params(axis='x', rotation=20)

pivot_lift = plot_df.pivot_table(index='데이터셋', columns='모델', values='Lift_at_25_mean', aggfunc='mean')
pivot_lift.plot(kind='bar', ax=axes[1])
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1, label='무작위 수준')
axes[1].set_title('피처셋 x 모델별 Lift@25% 평균')
axes[1].set_ylabel('Lift@25% (mean)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig2_featureset_comparison.png'), dpi=150)
plt.show()

print("해석: 'A_피부타입만'보다 B/C/D가 높으면 성분 정보가 기여한다는 뜻입니다.")
print("      B(핵심60)가 D(전성분1352)와 비슷하면 고차원 원핫은 불필요합니다.")

In [ ]:
# 3. 최종 우수 모델의 OOF Precision-Recall Curve
best_oof = oof_df_combined[
    (oof_df_combined['dataset'] == best_model_info['dataset']) &
    (oof_df_combined['model'] == best_model_info['model']) &
    (oof_df_combined['stage'] == best_model_info['stage'])
]
prec_curve, rec_curve, _ = precision_recall_curve(best_oof['high_risk'], best_oof['predicted_probability'])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(rec_curve, prec_curve, color='darkorange', lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f"최종 우수 모델 OOF Precision-Recall Curve\n"
             f"({best_model_info['dataset']} - {best_model_info['model']} - {best_model_info['stage']})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig3_best_model_pr_curve.png'), dpi=150)
plt.show()

In [ ]:
# 4. 최종 우수 모델 Confusion Matrix (OOF)
best_pred_label = (best_oof['predicted_probability'] >= best_oof['selected_threshold']).astype(int)
cm = confusion_matrix(best_oof['high_risk'], best_pred_label)

fig, ax = plt.subplots(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 14})
ax.set_xticklabels(['Safe (0)', 'Risk (1)'])
ax.set_yticklabels(['Safe (0)', 'Risk (1)'], va='center')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f"최종 우수 모델 Confusion Matrix (OOF)\n"
             f"({best_model_info['dataset']} - {best_model_info['model']})")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig4_best_model_confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# 5. Fold별 선택 임계값 분포
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=fold_df_combined, x='model', y='threshold', hue='dataset', ax=ax)
ax.set_title('Fold별 선택 임계값 분포 (모델 x 데이터셋)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig5_threshold_distribution.png'), dpi=150)
plt.show()

In [ ]:
# 6. Precision@Top25%, Lift@Top25% 비교
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(np.arange(len(plot_df)), plot_df['Precision_at_25_mean'], color='steelblue')
axes[0].set_xticks(np.arange(len(plot_df)))
axes[0].set_xticklabels(plot_labels, rotation=45, ha='right', fontsize=7)
axes[0].set_title('Precision@Top25% 비교')

axes[1].bar(np.arange(len(plot_df)), plot_df['Lift_at_25_mean'], color='salmon')
axes[1].set_xticks(np.arange(len(plot_df)))
axes[1].set_xticklabels(plot_labels, rotation=45, ha='right', fontsize=7)
axes[1].set_title('Lift@Top25% 비교')
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig6_top25_metrics.png'), dpi=150)
plt.show()

## 19-2. (보조 분석) 제품 단위 예측 모델

제품×피부타입 단위는 구조적 한계가 있습니다. 성분 변수는 한 제품의 7개 피부타입 행에
**모두 동일하게** 들어가는데, 부정률 변동의 상당 부분이 제품 내부(피부타입 간)에서 발생하기 때문입니다.

여기서는 피부타입 축을 접고 **제품 단위**로 부정률을 집계해 예측합니다.
- 제품당 리뷰가 훨씬 많아 라벨 노이즈가 줄어듭니다.
- 성분이 설명할 수 있는 '제품 간 변동'만 남으므로 과제와 피처가 정렬됩니다.
- 대신 "어떤 피부타입에서 위험한가"는 답할 수 없으므로, 메인 분석의 보조로 사용하세요.


In [ ]:
MIN_REVIEWS_PER_PRODUCT = 10   # 제품 단위 최소 리뷰 수

# ---- 제품 단위 Y ----
p_group = (
    df_review.groupby('product_id')
    .agg(y_전체리뷰=('리뷰분류_1_0', 'count'), y_긍정=('리뷰분류_1_0', 'sum'))
    .reset_index()
)
p_group['y_부정'] = p_group['y_전체리뷰'] - p_group['y_긍정']
p_group['y_부정비율'] = p_group['y_부정'] / p_group['y_전체리뷰']
p_group = p_group[p_group['y_전체리뷰'] >= MIN_REVIEWS_PER_PRODUCT].reset_index(drop=True)

print(f"제품 수: {len(p_group)}, 제품당 평균 리뷰: {p_group['y_전체리뷰'].mean():.0f}건")

# 라벨 신뢰도(관측분산 중 실제 신호 비율) 참고 출력
_p = p_group['y_부정'].sum() / p_group['y_전체리뷰'].sum()
_v_obs = p_group['y_부정비율'].var()
_v_noise = np.mean(_p * (1 - _p) / p_group['y_전체리뷰'])
print(f"제품 단위 라벨 신뢰도(1 - 노이즈/관측): {1 - _v_noise / _v_obs:.3f}")

# ---- 제품 단위 X: 피부타입과 무관한 변수만 (v3/v4/v5) ----
prod_x_base = df_v1.drop_duplicates('product_id').copy()
prod_merged = pd.merge(prod_x_base, p_group, on='product_id', how='inner')
prod_merged = prod_merged.sort_values('product_id').reset_index(drop=True)

prod_feat_cols = [c for c in prod_merged.columns if c.startswith(('v3_', 'v4_', 'v5_'))]
prod_safe_names, prod_mapping = make_safe_column_names(
    prod_feat_cols, os.path.join(RESULTS_DIR, 'feature_name_mapping_product_level.csv')
)
X_prod = prod_merged[prod_feat_cols].copy()
X_prod.columns = prod_safe_names

prod_threshold = prod_merged['y_부정비율'].quantile(RISK_QUANTILE)
y_prod = (prod_merged['y_부정비율'] >= prod_threshold).astype(int).values

assert X_prod.select_dtypes(exclude=[np.number]).shape[1] == 0, "제품 단위 X에 비숫자형 컬럼이 있습니다."
assert X_prod.isna().sum().sum() == 0, "제품 단위 X에 결측값이 있습니다."

print(f"제품 단위 학습 데이터: {X_prod.shape[0]}행 x {X_prod.shape[1]}피처")
print(f"고위험 기준 부정률: {prod_threshold:.4f} / 고위험 비율: {y_prod.mean():.3f}")

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split as _tts

def run_product_level_cv(X, y, model_names, n_splits=N_SPLITS, random_state=RANDOM_STATE):
    """제품 단위는 한 제품당 1행이므로 그룹 분할이 필요 없음 (StratifiedKFold 사용).
    임계값은 외부 Validation을 보지 않고 Train 내부 분할에서 선택."""
    records, oof_records = [], []
    cv_p = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold_i, (tr_idx, va_idx) in enumerate(cv_p.split(X, y)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        spw = n_neg / n_pos if n_pos > 0 else 1.0

        for model_name in model_names:
            ModelClass, params = get_model_spec(model_name, spw)
            model = ModelClass(**params)

            X_fit, X_thr, y_fit, y_thr = _tts(
                X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=random_state
            )
            model.fit(X_fit, y_fit)
            best_threshold = select_threshold(y_thr, model.predict_proba(X_thr)[:, 1])

            va_prob = model.predict_proba(X_va)[:, 1]
            va_pred = (va_prob >= best_threshold).astype(int)

            m = evaluate_predictions(y_va, va_prob, va_pred)
            m.update({'dataset': '제품단위_V345', 'model': model_name, 'stage': 'product_level',
                      'fold': fold_i, 'threshold': best_threshold,
                      'n_train': len(tr_idx), 'n_valid': len(va_idx), 'n_features': X.shape[1]})
            records.append(m)

            for local_i, global_i in enumerate(va_idx):
                oof_records.append({
                    'product_id': prod_merged['product_id'].values[global_i],
                    'product_name': prod_merged['product_name'].values[global_i],
                    'y_전체리뷰': prod_merged['y_전체리뷰'].values[global_i],
                    'y_부정비율': prod_merged['y_부정비율'].values[global_i],
                    'high_risk': int(y_va[local_i]),
                    'dataset': '제품단위_V345', 'model': model_name, 'fold': fold_i,
                    'predicted_probability': va_prob[local_i],
                    'selected_threshold': best_threshold,
                    'predicted_label': int(va_pred[local_i]),
                })

            print(f"[제품단위] Fold {fold_i} {model_name:18s} | Precision={m['Precision']:.3f} "
                  f"F0.5={m['F0.5']:.3f} PR-AUC={m['PR_AUC']:.3f} Lift@25%={m['Lift_at_25']:.2f}")

    return pd.DataFrame(records), pd.DataFrame(oof_records)


fold_df_prod, oof_df_prod = run_product_level_cv(X_prod, y_prod, BASE_MODEL_NAMES)
prod_summary = summarize_fold_metrics(fold_df_prod).sort_values('F05_mean', ascending=False).reset_index(drop=True)

print("\n[제품 단위 모델 성능]")
display(prod_summary)

fold_df_prod.to_csv(os.path.join(RESULTS_DIR, 'fold_metrics_product_level.csv'), index=False, encoding='utf-8-sig')


In [ ]:
oof_df_prod.to_csv(os.path.join(RESULTS_DIR, 'oof_predictions_product_level.csv'), index=False, encoding='utf-8-sig')
prod_summary.to_csv(os.path.join(RESULTS_DIR, 'model_comparison_product_level.csv'), index=False, encoding='utf-8-sig')

# 제품×피부타입 단위 최고 성능과 비교
best_skin_lift = summary_combined_f05['Lift_at_25_mean'].max()
best_prod_lift = prod_summary['Lift_at_25_mean'].max()
print(f"\n제품x피부타입 단위 최고 Lift@25%: {best_skin_lift:.2f}")
print(f"제품 단위 최고 Lift@25%      : {best_prod_lift:.2f}")
print("(1.00 = 무작위 수준. 제품 단위가 높다면 성분은 '제품 전체 위험도' 예측에 더 적합합니다.)")


In [ ]:
# 제품×피부타입 vs 제품 단위 성능 비교 그래프
fig, ax = plt.subplots(figsize=(11, 5))

cmp_rows = []
for _, r in summary_combined_f05.iterrows():
    cmp_rows.append({'구분': f"{r['데이터셋']}\n{r['모델']}", 'Lift@25%': r['Lift_at_25_mean'], '단위': '제품x피부타입'})
for _, r in prod_summary.iterrows():
    cmp_rows.append({'구분': f"제품단위\n{r['모델']}", 'Lift@25%': r['Lift_at_25_mean'], '단위': '제품 단위'})
cmp_df = pd.DataFrame(cmp_rows).sort_values('Lift@25%', ascending=False)

colors = ['tab:orange' if u == '제품 단위' else 'tab:blue' for u in cmp_df['단위']]
ax.bar(np.arange(len(cmp_df)), cmp_df['Lift@25%'], color=colors)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
ax.set_xticks(np.arange(len(cmp_df)))
ax.set_xticklabels(cmp_df['구분'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Lift@25%')
ax.set_title('제품x피부타입 단위(파랑) vs 제품 단위(주황) - Lift@25% 비교')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig7_unit_comparison.png'), dpi=150)
plt.show()

## 20. 최종 모델 저장

주의: 아래에서 학습하는 모델은 **전체 데이터로 재학습한 배포용 모델**입니다.
성능 판단은 반드시 교차검증 결과(`model_comparison_summary.csv`)를 기준으로 하세요.


In [ ]:
final_dataset_name = best_model_info['dataset']
final_model_name = best_model_info['model']
final_stage = best_model_info['stage']

final_ds = DATASETS[final_dataset_name]
X_final = final_ds['X']
feature_cols_final = final_ds['feature_cols_original']
safe_names_final = final_ds['safe_names']
mapping_df_final = final_ds['mapping']

tuned_params_final = tuned_params_all[final_dataset_name][final_model_name] if final_stage == 'tuned' else None

n_neg_full, n_pos_full = (y_all == 0).sum(), (y_all == 1).sum()
scale_pos_weight_full = n_neg_full / n_pos_full if n_pos_full > 0 else 1.0

ModelClass, params = get_model_spec(final_model_name, scale_pos_weight_full, tuned_params=tuned_params_final)

print("주의: 아래는 교차검증 성능이 아니라 전체 데이터로 재학습한 배포용(참고용) 모델입니다.")
print("교차검증 성능(Fold 평균)은 model_comparison_summary.csv를 기준으로 판단하세요.")

if final_model_name in EARLY_STOPPING_MODELS:
    gss_final = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
    fit_idx, val_idx = next(gss_final.split(X_final, y_all, groups=product_id_groups))
    assert len(set(product_id_groups[fit_idx]) & set(product_id_groups[val_idx])) == 0

    final_model = ModelClass(**params, early_stopping_rounds=30)
    if final_model_name == 'XGBoost':
        final_model.fit(X_final.iloc[fit_idx], y_all[fit_idx],
                        eval_set=[(X_final.iloc[val_idx], y_all[val_idx])], verbose=False)
    else:
        final_model.fit(X_final.iloc[fit_idx], y_all[fit_idx],
                        eval_set=(X_final.iloc[val_idx], y_all[val_idx]), use_best_model=True)
    final_threshold = select_threshold(y_all[val_idx], final_model.predict_proba(X_final.iloc[val_idx])[:, 1])
else:
    final_model = ModelClass(**params)
    final_threshold, _, final_model = fit_predict_with_group_threshold_split(
        final_model, X_final, y_all, product_id_groups, X_final
    )

print(f"최종 배포용 모델: {final_dataset_name} - {final_model_name} ({final_stage}), threshold={final_threshold:.3f}")

with open(os.path.join(RESULTS_DIR, 'final_model.pkl'), 'wb') as f:
    pickle.dump({
        'model': final_model, 'model_name': final_model_name, 'dataset_name': final_dataset_name,
        'stage': final_stage, 'threshold': final_threshold,
        'feature_columns_original': feature_cols_final, 'feature_columns_safe': safe_names_final,
        'column_mapping': mapping_df_final,
    }, f)

print("최종 모델 저장 완료: final_model.pkl")

## 21. 최종 신제품 예측 함수


In [ ]:
def predict_new_product_risk(new_product_feature_rows, trained_model, selected_threshold, feature_columns,
                             id_col='product_id', name_col='product_name', skin_col='피부타입'):
    """
    리뷰가 없는 신제품의 성분/피부타입 정보로 부정 경험 고위험군 여부를 예측.

    predicted_probability는 '실제 부정률'이 아니라,
    '과거 데이터에서 정의한 부정 경험 고위험군에 속할 모델 예측확률'을 의미합니다.

    Parameters
    ----------
    new_product_feature_rows : pd.DataFrame
        신제품 1개당 피부타입별로 여러 행(예: 7개)으로 구성된 데이터프레임.
        feature_columns에 해당하는 컬럼들과 id_col, name_col, skin_col을 포함해야 함.
    trained_model : 학습된 분류 모델 (predict_proba 지원)
    selected_threshold : float, 학습 시 선택된 임계값
    feature_columns : list, 모델 입력에 사용할 안전한 컬럼명 리스트

    Returns
    -------
    pd.DataFrame with columns: product_id, product_name, 피부타입,
        predicted_probability, predicted_label, risk_rank
    """
    missing_cols = [c for c in feature_columns if c not in new_product_feature_rows.columns]
    if missing_cols:
        raise ValueError(f"신제품 데이터에 다음 피처 컬럼이 없습니다: {missing_cols[:10]} ... (총 {len(missing_cols)}개)")

    X_new = new_product_feature_rows[feature_columns].copy()
    if X_new.select_dtypes(exclude=[np.number]).shape[1] > 0:
        raise ValueError("신제품 피처 데이터에 숫자형이 아닌 컬럼이 있습니다.")
    if X_new.isna().sum().sum() > 0:
        raise ValueError("신제품 피처 데이터에 결측값이 있습니다.")

    pred_prob = trained_model.predict_proba(X_new)[:, 1]
    pred_label = (pred_prob >= selected_threshold).astype(int)

    result = pd.DataFrame({
        'product_id': new_product_feature_rows[id_col].values,
        'product_name': (new_product_feature_rows[name_col].values
                         if name_col in new_product_feature_rows.columns else None),
        '피부타입': new_product_feature_rows[skin_col].values,
        'predicted_probability': pred_prob,
        'predicted_label': pred_label,
    })
    result['risk_rank'] = result['predicted_probability'].rank(ascending=False, method='min').astype(int)
    result = result.sort_values('risk_rank').reset_index(drop=True)

    print("[안내] predicted_probability의 의미:")
    print("  '해당 피부타입에서 실제로 나올 부정률'이 아니라,")
    print("  '과거 데이터에서 정의한 부정 경험 고위험군에 속할 모델 예측확률'입니다.")

    return result


print("predict_new_product_risk 함수 정의 완료.")


In [ ]:
print("""
사용 예시:
------------------------------------------------------------
import pickle
with open(EXPERIMENT_RESULTS_DIR / 'final_model.pkl', 'rb') as f:
    saved = pickle.load(f)

# 신제품 피처 데이터프레임은 학습 때와 동일한 방식으로 만들어야 합니다.
#  - product_id 1개당 피부타입 7행
#  - saved['feature_columns_safe']와 동일한 컬럼 구성

result = predict_new_product_risk(
    new_product_feature_rows=신제품_피처_데이터프레임,
    trained_model=saved['model'],
    selected_threshold=saved['threshold'],
    feature_columns=saved['feature_columns_safe'],
)
display(result)
------------------------------------------------------------
""")
